# Natural Language Autoencoders (NLAs)

**Survey Reference:** Natural Language Autoencoders — dedicated handbook section coming soon ([Anthropic, 2026](https://transformer-circuits.pub/2026/nla/))

**Prerequisites:** [01_logit_lens.ipynb](./01_logit_lens.ipynb), [06_activation_oracles.ipynb](./06_activation_oracles.ipynb)

The lenses in this section (logit / tuned / Jacobian) read an activation as a **distribution over single
vocabulary tokens**. A **Natural Language Autoencoder** ([Fraser-Taliente et al., 2026](https://transformer-circuits.pub/2026/nla/))
reads it as **free-form English** instead — an unsupervised explanation of what the activation encodes.

An NLA is a pair of fine-tuned LMs that autoencode a residual-stream vector *through a natural-language
bottleneck*:

| | direction | mechanism |
|---|---|---|
| **AV** — activation verbalizer | `vector → text` | inject the vector as a single token embedding into a fixed prompt, then autoregress a description |
| **AR** — activation reconstructor | `text → vector` | a truncated *K+1*-layer LM + `Linear(d, d)` head that maps the description back to a vector |

Training is unsupervised: the AV is rewarded when the AR can rebuild the original vector from the AV's
words alone (RL on `−‖h − AR(AV(h))‖²`). Both vectors are L2-normalised before comparison, so the
round-trip score is `MSE = 2(1 − cos)` — pure direction agreement. **Low MSE ⇔ the explanation captured
the vector's content.** The paper reports NLAs surface *safety-relevant cognition a model does not
verbalize* (e.g. unstated evaluation-awareness).

> **What this notebook does.** It follows the official inference path from
> [`kitft/natural_language_autoencoders`](https://github.com/kitft/natural_language_autoencoders):
> load a **released NLA** (Qwen2.5-7B, layer 20) and run the verbalizer (`NLAClient`) over an **SGLang
> server**, plus the reconstructor (`NLACritic`). It does **not** train an NLA — that needs 2×H100 (SFT)
> to 2×8×H100 (RL). Interactive exploration of released NLAs is on [Neuronpedia](https://www.neuronpedia.org/nla).

## 1. The SGLang server

The verbalizer injects an activation vector as a **token embedding** and autoregresses, so it must be
served with `input_embeds` (not `input_ids`). The official repo does this with **SGLang**. SGLang pins
its own torch/CUDA, so run it in an **isolated environment** — never in this notebook's kernel:

```bash
# one-time, in a fresh env (protects your notebook's torch/transformers)
pip install "sglang[all]>=0.5.6"

# launch the server on the released AV checkpoint (loads from the HF cache)
python -m sglang.launch_server \
    --model-path kitft/nla-qwen2.5-7b-L20-av \
    --port 30000 --disable-radix-cache \
    --trust-remote-code --context-length 512
```

`--disable-radix-cache` is **required**: the radix cache keys on token IDs, but `input_embeds` requests
have none, so different vectors would alias to the same cache entry and silently return garbage.

> **Troubleshooting** (pip-packaged CUDA, no system toolkit): SGLang JIT-compiles kernels at startup,
> so `ninja` and `nvcc` must be on `PATH` and the linker must find the CUDA libs. If you see
> `ninja: not found`, `-lcudart: No such file`, add the env's `bin/` and the
> `.../site-packages/nvidia/*/lib` dirs to `PATH`/`LIBRARY_PATH` and symlink the unversioned
> `libcudart.so → libcudart.so.13`.

The cell below just checks the server is reachable.

In [1]:
%pip install --quiet transformers safetensors pyyaml httpx orjson numpy huggingface_hub

import httpx
SGLANG_URL = "http://localhost:30000"
try:
    r = httpx.get(f"{SGLANG_URL}/health", timeout=5)
    print(f"SGLang server: {'UP' if r.status_code == 200 else r.status_code} at {SGLANG_URL}")
except Exception as e:
    raise RuntimeError(
        f"No SGLang server at {SGLANG_URL}. Launch it first (see the command above) "
        f"in a separate environment, then re-run this cell.\n  ({e})"
    )

Note: you may need to restart the kernel to use updated packages.


SGLang server: UP at http://localhost:30000


## 2. The official inference code

We use the single-file `nla_inference.py` from the repo, which provides:

- **`NLAClient`** — the verbalizer (AV). Loads only the AV's embedding table (not the full model — SGLang
  holds that), splices the activation into the prompt, and calls the server.
- **`NLACritic`** — the reconstructor (AR). Pure torch; loads the truncated backbone + value head and
  scores an explanation against the original vector.

One shim is needed: the official `NLAClient` tokenises its prompt with
`apply_chat_template(tokenize=True)`, which **regressed in transformers ≥5.x** (it returns a truncated
2-token sequence for this tokenizer, so the injection marker is lost). We subclass it and override *only*
the tokenisation to the equivalent `apply_chat_template(tokenize=False)` → `encode()` path. The
SGLang call (`_sglang_generate`) and the entire AR (`NLACritic`) are the official code, untouched.

In [2]:
import os, sys, subprocess, yaml, torch

REPO_DIR = "/tmp/nla_repo"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/kitft/natural_language_autoencoders", REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)

from nla_inference import (
    NLAClient, NLAConfig, NLACritic, load_embedding_only, resolve_embed_scale,
    normalize_activation, inject_at_marked_positions, INJECT_PLACEHOLDER,
)
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download


class NLAClientHF(NLAClient):
    "Official NLAClient with a transformers>=5 chat-template tokenisation fix."
    def __init__(self, checkpoint_dir, sglang_url):
        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir, trust_remote_code=True)
        m = yaml.safe_load(open(os.path.join(checkpoint_dir, "nla_meta.yaml")))
        t = m["tokens"]
        self.cfg = NLAConfig(
            d_model=m["d_model"], injection_char=t["injection_char"],
            injection_token_id=t["injection_token_id"],
            injection_left_neighbor_id=t["injection_left_neighbor_id"],
            injection_right_neighbor_id=t["injection_right_neighbor_id"],
            actor_prompt_template=m["prompt_templates"]["av"],
            injection_scale=float(m["extraction"]["injection_scale"]),
        )
        self.embed = load_embedding_only(checkpoint_dir, dtype=torch.bfloat16)
        self.embed_scale = resolve_embed_scale(checkpoint_dir)
        self.sglang_url = sglang_url.rstrip("/")
        self._http = httpx.Client(timeout=httpx.Timeout(180.0))

    def _build_embeds(self, v_raw, prompt_content):
        c = self.cfg
        content = (c.actor_prompt_template.format(injection_char=c.injection_char)
                   if prompt_content is None
                   else prompt_content.replace(INJECT_PLACEHOLDER, c.injection_char))
        s = self.tokenizer.apply_chat_template(
            [{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
        ids = self.tokenizer.encode(s, add_special_tokens=False)
        ids_t = torch.tensor(ids, dtype=torch.long).unsqueeze(0)
        with torch.no_grad():
            embeds = (self.embed(ids_t.to(self.embed.weight.device)) * self.embed_scale).float()
        v_scaled = normalize_activation(v_raw.float().view(1, -1), c.injection_scale)
        injected = inject_at_marked_positions(
            ids_t, embeds.cpu(), v_scaled,
            c.injection_token_id, c.injection_left_neighbor_id, c.injection_right_neighbor_id)
        return injected[0].contiguous().numpy(), len(ids)

print("NLAClientHF ready.")

/mnt/data0/user/conda_envs/irms/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NLAClientHF ready.


## 3. Load the verbalizer (AV)

`NLAClient` needs only the AV's **embedding table** and its `nla_meta.yaml` **sidecar** — the token IDs,
prompt template, and `injection_scale` the model was trained with. (The full AV weights live in the
SGLang server.) Never hardcode these; always read the sidecar.

In [3]:
AV_REPO = "kitft/nla-qwen2.5-7b-L20-av"
av_dir = snapshot_download(AV_REPO)
client = NLAClientHF(av_dir, SGLANG_URL)

print(f"d_model         : {client.cfg.d_model}")
print(f"injection_scale : {client.cfg.injection_scale}  (L2 norm vectors are rescaled to)")
print(f"injection marker: {client.cfg.injection_char!r}  (token id {client.cfg.injection_token_id})")

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Fetching 18 files: 100%|██████████| 18/18 [00:00<00:00, 2130.17it/s]

d_model         : 3584
injection_scale : 150.0  (L2 norm vectors are rescaled to)
injection marker: '㈎'  (token id 149705)


**Smoke test.** The repo's quickest check verbalises a *random* vector — it confirms the server responds
and the injection path runs. A random direction is out-of-distribution, so the text is loose and may
carry a few stray characters; that's expected. The tell-tale *failure* is output that is **entirely**
CJK — that means the vector was never injected and the model is describing the marker character `㈎`
itself (see the repo's debugging notes). Real activations (next section) decode cleanly.

In [4]:
import numpy as np

rand_v = np.random.randn(client.cfg.d_model).astype(np.float32)
print(client.generate(rand_v, temperature=0.7, max_new_tokens=200))

Formal math problem format with numbered steps and engineering notation, following a standard "Difficulty/Proof" pattern common to physics textbooks.

The phrase "定义 '计算矩阵' 的计算过程,表示大家不變的值" is a common LaTeX template pattern for "Problem Solution" with the final term "student's accumulated score" implying the famous "matrix" acronym for the original object being tested.

Final token "値" ends the "内聚 不变的内" phrase, part of the common "So the original students have no problem" idiom — expects " score" or "the matrix" or "the student" to complete the common "concentration" definition.


## 4. Verbalize a real activation

Now the real thing: run the **target model** (base Qwen2.5-7B-Instruct), grab its **layer-20** residual
at the last token, and ask the NLA what it encodes. Layer 20 is ~⅔ of the way up — deep enough to carry
rich semantics, shallow enough not to have collapsed onto the next-token prediction.

In [5]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LAYER = 20

base_tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16).cuda().eval()


def activation_at_last_token(text, layer=LAYER):
    ids = base_tok(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        hs = base(**ids, output_hidden_states=True).hidden_states[layer][0]
    return hs[-1].detach().float().cpu()   # [d_model]


prompt = ("The patient presented with a high fever, a persistent cough, and difficulty "
          "breathing, so the doctor ordered a chest")
v = activation_at_last_token(prompt)
print(f"activation norm = {v.norm():.1f}\n")
print("NLA explanation:\n")
print(client.generate(v.numpy(), temperature=0.7, max_new_tokens=180))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 339/339 [00:00<00:00, 10000.27it/s]

activation norm = 104.6

NLA explanation:



Medical/clinical context with a formal format suggesting a nursing or health question prompt, implying a patient scenario requiring medical procedures or symptoms.

The phrase "The physician ordered a chest" strongly implies a specific medical test or examination term follows, most likely "X-ray" or "X-ray examination," completing a patient complaint scenario about a chest condition or pneumonia diagnosis.

Final token "chest" is mid-phrase ("a chest"), part of a list ("The doctor ordered a chest"), immediately expecting a noun like "X-ray" or "exam," or "radiograph," or "cavity examination," continuing the medical procedure description about the patient's lung condition.


The NLA reads the residual as a **clinical / chest-imaging** context and anticipates the completion
(*chest X-ray*) — none of which is a single next-token readout. The model has built that content into the
residual stream, and the NLA verbalises it.

## 5. Close the loop — reconstruct and score (AR)

Is the explanation *faithful*, or a plausible guess? The **AR** answers this: it maps the explanation
text back to a vector, and we compare directions. `NLACritic.score` returns `(mse, cos)` with both
vectors L2-normalised, so `mse = 2(1 − cos)`:

`cos ≈ 0.9` → good decode · `0.5` → mediocre · `0.0` → orthogonal.

In [6]:
AR_REPO = "kitft/nla-qwen2.5-7b-L20-ar"
ar_dir = snapshot_download(AR_REPO)
critic = NLACritic(ar_dir, device="cuda")   # official code, unmodified

explanation = client.generate(v.numpy(), temperature=0.7, max_new_tokens=180)
mse, cos = critic.score(explanation, v)
print(f"reconstruction  cos = {cos:.3f}   mse = {mse:.3f}   (cos~0.9 = good decode)")

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Fetching 18 files: 100%|██████████| 18/18 [00:00<00:00, 2314.60it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/253 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 253/253 [00:00<00:00, 8845.28it/s]

[transformers] Qwen2ForCausalLM LOAD REPORT from: /mnt/data0/user/hf_cache/hub/models--kitft--nla-qwen2.5-7b-L20-ar/snapshots/e2c9e57eac213d37a31612087f645ab6332c1bb6
Key               | Status  | 
------------------+---------+-
lm_head.weight    | MISSING | 
model.norm.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[NLACritic] 21 layers  d_model=3584  mse_scale=59.87


reconstruction  cos = 0.888   mse = 0.225   (cos~0.9 = good decode)


## 6. Reading several activations

The same round-trip across a few prompts from different domains. The `cos` column is the AR's fidelity
check — how well the explanation's words let the AR rebuild the original direction.

In [7]:
PROMPTS = [
    "In 1969, NASA's Apollo 11 mission landed the first two humans on the surface of the",
    "A recipe for chocolate chip cookies: cream the butter and sugar, beat in the eggs, then fold in the flour and chocolate",
    "def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[len(arr)//2]\n    left = [x for x in arr if x < pivot]",
    "The orchestra tuned their instruments as the conductor raised his baton to begin the symphony",
]
for p in PROMPTS:
    v_i = activation_at_last_token(p)
    expl = client.generate(v_i.numpy(), temperature=0.7, max_new_tokens=160)
    _, cos_i = critic.score(expl, v_i)
    snippet = expl.split("\n\n")[0]
    print(f"[cos={cos_i:.2f}]  ...{p[-46:]!r}")
    print(f"          -> {snippet[:150]}\n")

[cos=0.89]  ...'ded the first two humans on the surface of the'
          -> Historical article format with bold headline structure ("The First Photographs of Apollo Moonwalkers") establishes an informational, formal tone about



[cos=0.86]  ...'the eggs, then fold in the flour and chocolate'
          -> Math problem format with numbered steps establishes a recipe context, implying a math problem describing a cake-baking process.



[cos=0.71]  ...')//2]\n    left = [x for x in arr if x < pivot]'
          -> Structured Python function format with defined variables and formatted list processing pattern, following a code documentation style with "def quickso



[cos=0.87]  ...'nductor raised his baton to begin the symphony'
          -> Formal English language context with a playful or imaginative prompt structure ("A student excitedly entering the hall"), suggesting a literary or edu



Each explanation tracks its domain (spaceflight, baking, sorting code, orchestral music) and typically
anticipates the continuation. Code (`quicksort`) scores lower — algorithmic structure is harder to pin
down in one direction than a concrete topic.

## 7. Limitations and where NLAs sit

- **Imperfect fidelity.** Released NLAs reach ~70–75% FVE; `cos` is usually 0.7–0.9, not 1.0. The
  explanation is the model's *gist* of the vector and can drift (a France-vs-Italy slip on a two-hop
  prompt is typical), so treat it as a hypothesis to check, not ground truth.
- **Context-dependent.** The extraction layer has to have seen enough left context (the repo uses a
  minimum position of ~50). Early-sequence activations decode toward the training prior.
- **Serving cost.** Explanations average ~130 tokens; inference is the binding constraint, and the
  official path needs an SGLang server per model.
- **Injection is fragile.** A tokenizer/template mismatch makes the AV describe the marker character
  itself — the loud CJK-output smoke test (§3) catches it.

### How it compares

| Method | Supervision | Reads an activation as | Bottleneck |
|---|---|---|---|
| Logit / Jacobian lens | none | one vocabulary token | the unembedding |
| Sparse autoencoder | unsupervised | weighted sum of learned atoms | a fixed feature dictionary |
| [Activation oracle](./06_activation_oracles.ipynb) | **supervised** | free-form text answer to a query | trained on ground-truth labels |
| **Natural Language Autoencoder** | **unsupervised** | free-form English explanation | **natural language**, scored by reconstruction |

An NLA is the unsupervised, free-text member of the family: no label set and no fixed dictionary — the
only training signal is "can the reconstructor rebuild the vector from your words?"

## References

- Fraser-Taliente, Kantamneni, Ong et al. (2026). **"Natural Language Autoencoders Produce Unsupervised
  Explanations of LLM Activations."** [Transformer Circuits Thread](https://transformer-circuits.pub/2026/nla/).
- [`kitft/natural_language_autoencoders`](https://github.com/kitft/natural_language_autoencoders) —
  official training + inference code, and the [`kitft/nla-models`](https://huggingface.co/collections/kitft/nla-models)
  released checkpoints (Qwen2.5-7B, Gemma-3-12B/27B, Llama-3.3-70B).
- [`ceselder/EasyNLA`](https://github.com/ceselder/EasyNLA) — a compact reimplementation (SFT + RL) of the method.
- [Natural Language Autoencoders](https://learnmechinterp.com/topics/natural-language-autoencoders/) —
  *Learn MI* tutorial.
- [Neuronpedia · NLA](https://www.neuronpedia.org/nla) — interactive frontend for the released NLAs.